In [1]:
from datetime import date
from pathlib import Path

import pandas as pd
import polars as pl
import yaml

from build_hexagon_network import (
    aggregate_by_device,
    aggregate_by_trip,
    build_conncections,
    get_daily_stop_sequence_pl,
)

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [2]:
RES = 7
VARIANT = "v2"  # of stop detection

In [3]:
path = f"{config['dirs']['output']}/{config['city']}"
Path(f"{path}/hexagon_network/{VARIANT}/{RES}").mkdir(exist_ok=True, parents=True)

In [4]:
dummy_seq = pd.DataFrame(
    {
        "device_id": [1] * 7,
        "day": [date(2026, 5, 19)] * 5 + [date(2026, 5, 20)] * 2,
        "h3_7": [1, 2, 3, 2, 3, 1, 2],
        "latitude_mean": [47.42, 47.64, 47.23, 47.87, 47.25, 47.42, 47.64],
        "longitude_mean": [18.91, 19.12, 19.09, 19.31, 19.08, 18.91, 19.12],
    }
)

get_daily_stop_sequence_pl(pl.from_pandas(dummy_seq), "h3_7")

{2: {3: 2}, 3: {2: 1, 1: 1}, 1: {2: 2}}

In [5]:
connections = build_conncections(path, RES, VARIANT)


In [6]:
connections

device_id,day,hour,minute,source,target,weight,dow
i64,str,i8,i8,str,str,u32,i8
1,"""2024-10-05""",11,6,"""871e03785ffffff""","""871e0378cffffff""",1,6
1,"""2024-10-07""",5,35,"""871e0378cffffff""","""871e036a6ffffff""",1,1
1,"""2024-10-07""",9,22,"""871e036a6ffffff""","""871e0378cffffff""",1,1
1,"""2024-10-09""",4,46,"""871e0378cffffff""","""871e03634ffffff""",1,3
1,"""2024-10-09""",9,12,"""871e03634ffffff""","""871e03726ffffff""",1,3
…,…,…,…,…,…,…,…
1652702,"""2025-12-20""",4,47,"""871e036a3ffffff""","""871e036a1ffffff""",1,6
1652702,"""2025-12-20""",5,10,"""871e036a1ffffff""","""871e036a2ffffff""",1,6
1652702,"""2025-12-20""",5,36,"""871e036a2ffffff""","""871e036a3ffffff""",1,6


In [9]:
connections

device_id,day,hour,source,target,weight,dow
i64,str,i8,str,str,u32,i8
1,"""2024-10-05""",11,"""871e03785ffffff""","""871e0378cffffff""",1,6
1,"""2024-10-07""",5,"""871e0378cffffff""","""871e036a6ffffff""",1,1
1,"""2024-10-07""",9,"""871e036a6ffffff""","""871e0378cffffff""",1,1
1,"""2024-10-09""",4,"""871e0378cffffff""","""871e03634ffffff""",1,3
1,"""2024-10-09""",9,"""871e03634ffffff""","""871e03726ffffff""",1,3
…,…,…,…,…,…,…
1652702,"""2025-12-20""",4,"""871e036a3ffffff""","""871e036a1ffffff""",1,6
1652702,"""2025-12-20""",5,"""871e036a1ffffff""","""871e036a2ffffff""",1,6
1652702,"""2025-12-20""",5,"""871e036a2ffffff""","""871e036a3ffffff""",1,6


In [ ]:
connections.write_parquet(f"{path}/hexagon_network/{VARIANT}/{RES}/connections.parquet")

In [7]:
network_by_trip = aggregate_by_trip(connections)

In [ ]:
network_by_trip.write_csv(f"{path}/hexagon_network/{VARIANT}/{RES}/edges_trip.csv")

In [ ]:
network_by_unique_device = aggregate_by_device(connections)
network_by_unique_device.write_csv(
    f"{path}/hexagon_network/{VARIANT}/{RES}/edges_device.csv"
)